# Semana 2: Tarea

**Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)**
**Entrega: hasta el lunes de la semana siguiente, vía `git commit` + `push` en tu fork** (`02_costo_capital_wacc/clase02_tarea.ipynb`)

Completa las celdas marcadas con `# TU CÓDIGO AQUÍ` y las celdas de texto marcadas con .
El notebook debe correr de inicio a fin sin errores (`Kernel  Restart & Run All`).

## Parte 1: beta de tu empresa (5 pts)

Elige una empresa listada distinta de SCCO (sugerencias: BVN, IFS, o una que te interese) y estima su beta
contra el S&P 500 con 5 años de retornos mensuales. Reporta beta, error estándar y $R^2$.

In [1]:
import yfinance as yf
import statsmodels.api as sm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKER = "MELI"   #  tu empresa

# TU CÓDIGO AQUÍ: descarga, retornos, regresión y summary

px = yf.download(["MELI", "^GSPC"], start="2021-08-01",
                 interval="1mo", auto_adjust=True, progress=False)["Close"]
r = px.pct_change().dropna()
r = r.rename(columns={"^GSPC": "SP500"})
X = sm.add_constant(r["SP500"])
modelo = sm.OLS(r["MELI"], X, missing="drop").fit()
print(modelo.summary())

beta_ols = modelo.params["SP500"]
ee = modelo.bse["SP500"]
print(f"\nBeta OLS = {beta_ols:.3f}  (error estándar {ee:.3f})")
print(f"Intervalo aproximado al 95%: [{beta_ols - 2*ee:.2f}, {beta_ols + 2*ee:.2f}]")

                            OLS Regression Results                            
Dep. Variable:                   MELI   R-squared:                       0.240
Model:                            OLS   Adj. R-squared:                  0.227
Method:                 Least Squares   F-statistic:                     18.62
Date:                Sun, 13 Sep 2026   Prob (F-statistic):           6.16e-05
Time:                        20:51:00   Log-Likelihood:                 50.864
No. Observations:                  61   AIC:                            -97.73
Df Residuals:                      59   BIC:                            -93.51
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0055      0.014     -0.395      0.6

 **Interpreta en una oración cada uno** (estilo CFA, *holding all else constant*):

- Beta: Ante un incremento de 1% en el retorno del S&P 500, se espera que el retorno de MELI aumente en 1.313%.
- Error estándar: El valor de 0.304 genera un intervalo de confianza amplio (0.704 y 1.922) que implica incertidumbre sobre el valor del beta.
- $R^2$: Un $R^2$ de 0.240 indic que solo el 24% e la variaabilidad de los retornos MELI es explicado por los movimientos de S&P500.

## Parte 2: beta ajustado y costo del equity (4 pts)

Calcula el beta ajustado de Blume y el $K_e$ por CAPM. Declara tu tasa libre de riesgo (fuente) y tu ERP (justifícala en una línea).

In [2]:
def beta_ajustado(beta_ols):
    # TU CÓDIGO AQUÍ
    return 0.67 * beta_ols + 0.33

beta_aj = beta_ajustado(beta_ols)
print(f"Beta OLS      = {beta_ols:.3f}")
print(f"Beta ajustado = {beta_aj:.3f}")

def capm(rf, beta, erp, crp=0.0, lam=1.0):
    # TU CÓDIGO AQUÍ
    return rf + beta * erp + lam * crp

import sys; sys.path.append("..")
from utils import fred
dgs10 = fred.get_series("DGS10").dropna()
rf = float(dgs10.iloc[-1]) / 100
print(f"Tasa libre de riesgo (T10Y, FRED): {rf:.2%}")

ERP = 0.055
ke = capm(rf, beta_aj, ERP)
print(f"Costo del equity (CAPM) = {ke:.2%}")

# TU CÓDIGO AQUÍ: rf, ERP y Ke con tus supuestos declarados
print ("Supuestos CAPM:")
print (f" rf = {rf:.2%} (fuente: FRED, serie DGS10, T10Y)")
print (f" ERP = {ERP:.2%} (fuente: Damodaran, prima de riesgo de mercado USA)")
print (f" Ke = {ke:.2%} (CAPM: rf + beta_aj * ERP)")

Beta OLS      = 1.313
Beta ajustado = 1.210
Tasa libre de riesgo (T10Y, FRED): 4.95%
Costo del equity (CAPM) = 11.60%
Supuestos CAPM:
 rf = 4.95% (fuente: FRED, serie DGS10, T10Y)
 ERP = 5.50% (fuente: Damodaran, prima de riesgo de mercado USA)
 Ke = 11.60% (CAPM: rf + beta_aj * ERP)


Como nota, no estoy incorporando el riesgo país ni el λ al cálculo del CAPM aunque sea una empresa que opere en latinoamérica, lo que sería de importancia en un ejercicio real.

## Parte 3: prima por riesgo país (3 pts)

Repite el cálculo de $K_e$ agregando una prima por riesgo país de 1.6% con $\lambda = 1$.

In [3]:
# TU CÓDIGO AQUÍ: Ke con CRP
CRP = 0.016
def capm(rf, beta, erp, crp=0.016, lam=1.0):
    return rf + beta * erp + lam * crp
ke = capm(rf, beta_aj, ERP)
print(f"Costo del equity (CAPM) = {ke:.2%}")

Costo del equity (CAPM) = 13.20%


 **Responde:** un proyecto de esta empresa rinde 12%. ¿Se acepta con el $K_e$ sin riesgo país? ¿Y con riesgo país?
¿Qué aprendes sobre valorar en mercados emergentes?

*Se acepta el $K_e$ sin riesgo país?*

Sí, el proyecto rinde a un 12% que es mayor al $K_e$ sin riesgo país.

*Y con riesgo país?*

No, con el CRP, el $K_e$ se eleva a 13.2% lo que hace que el rendimiento no cubra el costo de  capital ajustado.

*Qué aprendes sobre valorar en mercados emergentes?*

Es importante tener en cuenta el riesgo país en proyectos que operan en mercados emergentes. El riesgo adicional encarecen estos proyectos volviendolos menos rentables que otros que operan en situciones con menor riesgo.

## Parte 4: el WACC completo (4 pts)

Construye el WACC de tu empresa con datos de `yfinance`: capitalización bursátil, deuda total del balance,
$K_d$ aproximado por gasto de intereses y la tasa de impuestos que corresponda a su jurisdicción (decláralo).
Verifica el control de razonabilidad: $K_d(1-t) < WACC < K_e$.

In [4]:
def wacc(E, D, ke, kd, t):
    # TU CÓDIGO AQUÍ
    V = E + D
    return E / V * ke + D / V * kd * (1 - t)


tk = yf.Ticker("MELI")
E = tk.fast_info["marketCap"]

bs = tk.balance_sheet
fin = tk.financials
D = float(bs.loc["Total Debt"].iloc[0])

gasto_intereses = abs(float(fin.loc["Interest Expense"].iloc[0]))
kd = gasto_intereses / D
tax = 0.3   # empresa con operacion principal en Mexico (IR 30%)

print(f"E (market cap)     = {E/1e9:,.1f} mil millones USD")
print(f"D (deuda total)    = {D/1e9:,.1f} mil millones USD")
print(f"Kd aproximado      = {kd:.2%}")
print(f"WACC de MELI       = {wacc(E, D, ke, kd, tax):.2%}")

# TU CÓDIGO AQUÍ: E, D, kd, tax y WACC con supuestos declarados
print("Supuestos WACC:")
print(f"  E   = {E/1e9:,.1f} mil millones USD (fuente: yfinance, market cap actual)")
print(f"  D   = {D/1e9:,.1f} mil millones USD (fuente: yfinance, Total Debt balance)")
print(f"  Kd  = {kd:.2%}  (fuente: yfinance, gasto por intereses / deuda total)")
print(f"  tax = {tax:.1%}  (ISR corporativo federal de México, tasa vigente para personas morales)")
print(f"  WACC = {wacc(E, D, ke, kd, tax):.2%}")

E (market cap)     = 96.2 mil millones USD
D (deuda total)    = 11.4 mil millones USD
Kd aproximado      = 1.40%
WACC de MELI       = 11.91%
Supuestos WACC:
  E   = 96.2 mil millones USD (fuente: yfinance, market cap actual)
  D   = 11.4 mil millones USD (fuente: yfinance, Total Debt balance)
  Kd  = 1.40%  (fuente: yfinance, gasto por intereses / deuda total)
  tax = 30.0%  (ISR corporativo federal de México, tasa vigente para personas morales)
  WACC = 11.91%


In [ ]:
kd_after_tax = kd * (1 - tax)

print(f"Kd(1-t)              = {kd_after_tax:.2%}")
print(f"WACC de MELI         = {wacc(E, D, ke, kd, tax):.2%}")
print(f"Ke                   = {ke:.2%}")

Kd(1-t)             = 0.98%
WACC de MELI         = 11.91%
Ke                   = 13.20%


Se cumple que $K_d(1-t) < WACC < K_e$.

## Parte 5: ítems tipo CFA (4 pts)

Responde en la celda final, justificando en una línea cada respuesta.

**1.** Para valorar en dólares los flujos de largo plazo de una empresa, la tasa libre de riesgo *más apropiada* es:
&nbsp;&nbsp;A. la tasa de política monetaria de la Fed.
&nbsp;&nbsp;B. el rendimiento del Treasury a 3 meses.
&nbsp;&nbsp;C. el rendimiento del Treasury a 10 años.

**2.** Una acción tiene $\beta = 0.8$. Si el mercado sube 10% en un mes, el modelo predice que la acción *más probablemente*:
&nbsp;&nbsp;A. subirá exactamente 8%.
&nbsp;&nbsp;B. subirá alrededor de 8% en promedio, más su componente propio.
&nbsp;&nbsp;C. subirá 10% menos la tasa libre de riesgo.

**3.** Al calcular los pesos del WACC, un analista usa los valores en libros de deuda y patrimonio. Su WACC *más probablemente* queda:
&nbsp;&nbsp;A. correcto, porque el balance está auditado.
&nbsp;&nbsp;B. distorsionado, porque los pesos deben ser a valor de mercado.
&nbsp;&nbsp;C. distorsionado solo si la empresa no paga impuestos.

**4.** Con $K_d = 8\%$ y $t = 25\%$, el costo de la deuda después de impuestos es:
&nbsp;&nbsp;A. 2.0%
&nbsp;&nbsp;B. 6.0%
&nbsp;&nbsp;C. 8.0%

 **Respuestas:**

1. C. Se utiliza el rendimiento del Tresury a 10 años porque representa un horizonte similar a los flujos que se esperan estimar.
2. B. El beta mide la sensibilidad al mercado (0.8 x 10% = 8%) que es lo que se espera que suba sin tener en cuenta el componente interno de la empresa.
3. B. Los valores a evaluar deben ser de mercado según la formula del WACC.
4. B. $K_d(1-t)$ = 8% * (1 - 0.25) = 6%

---
**Recuerda:** `Kernel  Restart & Run All` antes de entregar, y luego:

```bash
git add 02_costo_capital_wacc/clase02_tarea.ipynb
git commit -m "Semana 2: tarea"
git push
```